## DDR PII Detection & Masking – Hands-On Tutorial

This 10-15 min notebook demonstrates the core Detection, Masking endpoints and threshold management using a small, production-grade client.  
**Endpoints**:  
    > `POST /api/v1/DetectPiiInText`  
    > `POST /api/v1/DetectPiiInFile`  
    > `POST /api/v1/MaskPiiInText`  
    > `POST /api/v1/MaskPiiInFile`  
    > `GET  /api/v1/thresholds`  
    > `POST /api/v1/thresholds`  

While this Tutorial focuses on PDF and Text, here are the **supported file types**  
>- PDF: pdf
>- Word: docx, docm, doc
>- Excel: xlsx, xlsm, xls, xltx, xltm
>- PowerPoint: pptx, pptm, ppt, pps, potx, potm, pot, ppsx, ppsm
>- Archive: zip, gzip, tar, 7z, rar, gz (Endpoint not demonstrated in this tutorial)
>- Images: png, bmp, jpg, jpeg, tif, tiff (Endpoint not demonstrated in this tutorial)
>- Eml : eml
>- Text : txt 

### Imports

In [1]:
import os
import sys
from pathlib import Path
from pprint import pprint
sys.path.append(os.path.abspath("../src"))

from pii_client import PiiClient, ApiError

### Initialize a PII Client

In the code block below we initialize the client with your auth key. make sure to **set it up as an ENV VAR before launching jupyter notebook** in the terminal.

In [2]:
BASE_URL = "https://prod.us.paralus.votiro.com/pii"
AUTH_TOKEN = os.environ["AUTH_KEY"]

client = PiiClient(base_url=BASE_URL, auth_token=AUTH_TOKEN)
print("Health:", client.health_check())

Health: True


## Supported Categories

Below you can find a list of **55 supported categories, devided into PII, PCI and PHI**. 

Each of the detection and masking endpoints allows for **fine grain control** over which categories will be detected and masked as needed.
We also provide the option to control the sensitivity of the detection of each category, through it's setting each category threshold based on your needs.

Upon completion of integration and with files flowing in, we employ our **proprietary solution to automatically adjust category threshold** based on each tenant's data distribution to maximize the quality of the detection and to reduce both False Positives and False Negatives.

In [3]:
PiiClient.show_labels_summary()

Category,Label ID,Label Name
PCI,52,BankAccount
PCI,53,CreditCard
PCI,54,CreditCardExpiration
PCI,55,Cvv
PCI,56,RoutingNumber
PHI,12,HealthcareNumber
PHI,31,OrganizationMedicalFacility
PHI,36,PhysicalAttribute
PHI,45,BloodType
PHI,46,Condition


#### Convert Label ID to Name and Vice Versa

In [4]:
label_id = 43
label_name = PiiClient.label_name_from_id(label_id)
print('Use func: PiiClient.label_name_from_id to convert from label_id to label_name')
print(f'E.g. label_id: {label_id} ==> label_name: {label_name}')


label_name = 'PoliticalAffiliation'
label_id = PiiClient.resolve_label_id(label_name)
print('\nUse func: PiiClient.resolve_label_id to convert from label_name to label_id')
print(f'E.g. label_name: {label_name} ==> label_id: {label_id}')

Use func: PiiClient.label_name_from_id to convert from label_id to label_name
E.g. label_id: 43 ==> label_name: VehicleId

Use func: PiiClient.resolve_label_id to convert from label_name to label_id
E.g. label_name: PoliticalAffiliation ==> label_id: 37


## Detect PII in Text

In the code block below we will use *client.detect_pii_in_text(text)* to POST text to this endpoint */api/v1/DetectPiiInText* and print out the results

In [5]:
text = (
    "Hi, I'm Jane Doe. My email is jane.doe@example.com and my phone is (415) 555-2671. "
    "My social security num is 123-45-6789. "  # PII (SSN)
    "My credit card number is 4111 1111 1111 1111, exp 12/27, and CVV 123. "  # PCI
    "My bank account number is 987654321 and routing number 021000021. "      # PCI
    "I was treated for diabetes and prescribed Metformin 500mg last year."    # PHI
)

text_result = client.detect_pii_in_text(text, human_readable_response=True)
pprint(text_result.get('piiFindings', {}))

[{'findingDuration': None,
  'labels': [{'confidence': 0.9092,
              'detectedByRegex': False,
              'label': 24,
              'labelName': 'Name'},
             {'confidence': 0.4652,
              'detectedByRegex': False,
              'label': 25,
              'labelName': 'NameFamily'},
             {'confidence': 0.4496,
              'detectedByRegex': False,
              'label': 26,
              'labelName': 'NameGiven'}],
  'text': 'Jane Doe'},
 {'findingDuration': None,
  'labels': [{'confidence': 0.8988,
              'detectedByRegex': False,
              'label': 8,
              'labelName': 'EmailAddress'}],
  'text': 'jane.doe@example.com'},
 {'findingDuration': None,
  'labels': [{'confidence': 0.9107,
              'detectedByRegex': False,
              'label': 35,
              'labelName': 'PhoneNumber'}],
  'text': '(415) 555-2671'},
 {'findingDuration': None,
  'labels': [{'confidence': 1.0,
              'detectedByRegex': True,
          

In the code block above we detected all categories with any confidence score - **this is the default behavior for this endpoint**.

**NOTICE**: Upon completion of integration and with files flowing in, we employ our **proprietary solution to automatically adjust category threshold** based on each tenant's data distribution to maximize the quality of the detection and reduction of both False Positives and False Negatives.

**NOTICE**: some instances will be detected for various categories with different confidence levels - that means that an **instance might have more than one category**. e.g. Name, NameFamily, NameGiven.

## Mask PII in Text

In the code block below we will use *client.mask_pii_in_text(text)* to POST text to this endpoint */api/v1/MaskPiiInText* and print out the results

In [6]:
masked = client.mask_pii_in_text(text) # default: mask all PII, PHI, PCI categories
print(masked)

﻿Hi, I'm ********. My email is ******************** and my phone is **************. My social security num is ***********. My credit card number is *******************, exp *****, and CVV ***. My bank account number is ********* and routing number *********. I was treated for ******** and prescribed ********* ***** last year.


#### Mask only PII Categories

In [7]:
only_pii_masked = client.mask_pii_in_text(text, include_pii=True) # mask only PII categories
print(only_pii_masked)

﻿Hi, I'm ********. My email is ******************** and my phone is **************. My social security num is ***********. My credit card number is 4111 1111 1111 1111, exp 12/27, and CVV 123. My bank account number is 987654321 and routing number 021000021. I was treated for diabetes and prescribed Metformin 500mg last year.


#### Mask only PCI Categories

In [8]:
only_pci_masked = client.mask_pii_in_text(text, include_pci=True) # mask only PCI categories
print(only_pci_masked)

﻿Hi, I'm Jane Doe. My email is jane.doe@example.com and my phone is (415) 555-2671. My social security num is ***-45-6789. My credit card number is *******************, exp *****, and CVV ***. My bank account number is ********* and routing number *********. I was treated for diabetes and prescribed Metformin 500mg last year.


#### Mask only PHI Categories

In [9]:
only_phi_masked = client.mask_pii_in_text(text, include_phi=True) # mask only PHI categories
print(only_phi_masked)

﻿Hi, I'm Jane Doe. My email is jane.doe@example.com and my phone is (415) 555-2671. My social security num is 123-45-6789. My credit card number is 4111 1111 1111 1111, exp 12/27, and CVV 123. My bank account number is 987654321 and routing number 021000021. I was treated for ******** and prescribed ********* ***** last year.


#### Mask only Specific Categories of Interest

In [10]:
include_labels=[8, 35]

print(f'masking only categories: {[PiiClient.label_name_from_id(label_id) for label_id in include_labels]}')

print(client.mask_pii_in_text(text, labels=include_labels))

masking only categories: ['EmailAddress', 'PhoneNumber']
﻿Hi, I'm Jane Doe. My email is ******************** and my phone is **************. My social security num is 123-45-6789. My credit card number is 4111 1111 1111 1111, exp 12/27, and CVV 123. My bank account number is 987654321 and routing number 021000021. I was treated for diabetes and prescribed Metformin 500mg last year.


## Detect PII in File (multipart upload)

In the code block below we will use *client.detect_pii_in_file(file_path)* to POST a file to this endpoint */api/v1/MaskPiiInFile* and print out the results

In [11]:
file_path = Path("../../query_files") / "tags" / "Invoice_Example.pdf"

file_result = client.detect_pii_in_file(file_path, human_readable_response=True)
pprint(file_result.get('piiFindings', {}))

[{'findingDuration': None,
  'labels': [{'confidence': 0.8663,
              'detectedByRegex': False,
              'label': 30,
              'labelName': 'Organization'}],
  'text': 'TechNova Analytics Ltd'},
 {'findingDuration': None,
  'labels': [{'confidence': 0.9004,
              'detectedByRegex': False,
              'label': 16,
              'labelName': 'LocationAddress'},
             {'confidence': 0.8735,
              'detectedByRegex': False,
              'label': 15,
              'labelName': 'Location'},
             {'confidence': 0.261,
              'detectedByRegex': False,
              'label': 17,
              'labelName': 'LocationCity'},
             {'confidence': 0.2505,
              'detectedByRegex': False,
              'label': 57,
              'labelName': 'LocationAddressStreet'},
             {'confidence': 0.1741,
              'detectedByRegex': False,
              'label': 21,
              'labelName': 'LocationZip'},
             {'confi

## Mask PII in File

In the code block below we will use *client.mask_pii_in_file(text)* to POST a file to this endpoint */api/v1/MaskPiiInFile*  

We save the mased content. Clicking the link, will open the masked file in a new tab.

In [12]:
content = client.mask_pii_in_file(file_path)
output_folder = "../../downloads"

# Save exactly as returned and show a clickable link
saved_path = PiiClient.save_and_link(
    content,
    filename="Invoice_Example_masked.pdf",   # keep .pdf so OS knows how to open
    directory=output_folder                    # change/omit as you like
)
print("Saved to:", saved_path)

/Users/aviavidan/src/votiro/ddr_turorials/downloads/Invoice_Example_masked-1.pdf

Saved to: ../../downloads/Invoice_Example_masked-1.pdf


For simpler comparison we can also visualize the origianl and masked files here in the notebook

In [13]:
PiiClient.preview_file(file_path)

In [14]:
PiiClient.preview_file(saved_path)

Just as we demonstrated above for text, the */api/v1/MaskPiiInFile* endpoint and correspondingly client.mask_pii_in_file() support the same filters to mask categories for  
> - PII only
> - PCI only
> - PHI only
> - Any combination of the above, or,
> - A list of specific categories

We do not show this here, but feel free to experiment with it yourself

## Modify Category Thresholds – GET & POST

We allow each tenant to manage the sensitivity of detection for each category by modifiying category thresholds.

**NOTICE**: 
> - We strongly recommend to use our data-driven solution for this, based on data distribution.
> - Changing the category thresholding, will affect any future detection that use_tenant_threshold (default behavior)


In the code block below we will use *client.get_thresholds()* to GET tenant category thresholds from this endpoint */api/v1/thresholds*  

#### Get Current Category Thresholds

In [15]:
current = client.get_thresholds()
print("Current thresholds (sample):")
[c for c in current['thresholds'] if current.get('tenantId') == client.tenant_id]

Current thresholds (sample):


[{'label': 25, 'minimalConfidence': 0.0, 'labelName': 'NameFamily'},
 {'label': 26, 'minimalConfidence': 0.0, 'labelName': 'NameGiven'},
 {'label': 1, 'minimalConfidence': 0.75, 'labelName': 'AccountNumber'},
 {'label': 2, 'minimalConfidence': 0.75, 'labelName': 'Age'},
 {'label': 3, 'minimalConfidence': 0.75, 'labelName': 'Date'},
 {'label': 4, 'minimalConfidence': 0.75, 'labelName': 'DateInterval'},
 {'label': 5, 'minimalConfidence': 0.75, 'labelName': 'Dob'},
 {'label': 6, 'minimalConfidence': 0.75, 'labelName': 'DriverLicense'},
 {'label': 7, 'minimalConfidence': 0.75, 'labelName': 'Duration'},
 {'label': 8, 'minimalConfidence': 0.75, 'labelName': 'EmailAddress'},
 {'label': 9, 'minimalConfidence': 0.75, 'labelName': 'Event'},
 {'label': 10, 'minimalConfidence': 0.75, 'labelName': 'Filename'},
 {'label': 12, 'minimalConfidence': 0.75, 'labelName': 'HealthcareNumber'},
 {'label': 13, 'minimalConfidence': 0.75, 'labelName': 'IpAddress'},
 {'label': 14, 'minimalConfidence': 0.75, 'lab

Above we can see if any categories has custom thresholds.  
**NOTICE**: **the default for these endpoints is to use threshold of 0.75 for all categories** - consider that this setting might not be ideal for your use case.  

As we saw above, the instance *Jane Doe*, got a few categories predicted with different confidence scores.

In [16]:
pii_inference = text_result.get('piiFindings', {})[0]
instance, categories = pii_inference.get('text'), pii_inference.get('labels', [])

print(f'lets zoom in on one instance detected for the text example:\n{text}\n')
print(f'categories predicted for instance: {instance}')
for cat in categories:
    for k, v in cat.items():
        print(f'{k}: {v}')

lets zoom in on one instance detected for the text example:
Hi, I'm Jane Doe. My email is jane.doe@example.com and my phone is (415) 555-2671. My social security num is 123-45-6789. My credit card number is 4111 1111 1111 1111, exp 12/27, and CVV 123. My bank account number is 987654321 and routing number 021000021. I was treated for diabetes and prescribed Metformin 500mg last year.

categories predicted for instance: Jane Doe
labelName: Name
label: 24
confidence: 0.9092
detectedByRegex: False
labelName: NameFamily
label: 25
confidence: 0.4652
detectedByRegex: False
labelName: NameGiven
label: 26
confidence: 0.4496
detectedByRegex: False


#### Modify Category Thresholds

If required we can modify the thresholds for specific categories, by using *client.post_thresholds()* which POSTs to */api/v1/thresholds*  

**IMPORTANT**: This will impact any future inference with *use_tenant_threshold=True* which the default in production DDR integration

In [17]:
update_category_thresholds = {'25': 0.75, '26': 0.75}
client.post_thresholds(thresholds=update_category_thresholds)
print("categories updated")

current = client.get_thresholds()
print("Current thresholds (sample):")
[c for c in current['thresholds'] if current.get('tenantId') == client.tenant_id]

categories updated
Current thresholds (sample):


[{'label': 25, 'minimalConfidence': 0.75, 'labelName': 'NameFamily'},
 {'label': 26, 'minimalConfidence': 0.75, 'labelName': 'NameGiven'},
 {'label': 1, 'minimalConfidence': 0.75, 'labelName': 'AccountNumber'},
 {'label': 2, 'minimalConfidence': 0.75, 'labelName': 'Age'},
 {'label': 3, 'minimalConfidence': 0.75, 'labelName': 'Date'},
 {'label': 4, 'minimalConfidence': 0.75, 'labelName': 'DateInterval'},
 {'label': 5, 'minimalConfidence': 0.75, 'labelName': 'Dob'},
 {'label': 6, 'minimalConfidence': 0.75, 'labelName': 'DriverLicense'},
 {'label': 7, 'minimalConfidence': 0.75, 'labelName': 'Duration'},
 {'label': 8, 'minimalConfidence': 0.75, 'labelName': 'EmailAddress'},
 {'label': 9, 'minimalConfidence': 0.75, 'labelName': 'Event'},
 {'label': 10, 'minimalConfidence': 0.75, 'labelName': 'Filename'},
 {'label': 12, 'minimalConfidence': 0.75, 'labelName': 'HealthcareNumber'},
 {'label': 13, 'minimalConfidence': 0.75, 'labelName': 'IpAddress'},
 {'label': 14, 'minimalConfidence': 0.75, 'l

you can see in the output of the cell above that the minimalConfidence is now changed 

In [18]:
text_result = client.detect_pii_in_text(text, human_readable_response=True, use_tenant_threshold=True)
pprint(text_result.get('piiFindings', {}))

[{'findingDuration': None,
  'labels': [{'confidence': 0.9092,
              'detectedByRegex': False,
              'label': 24,
              'labelName': 'Name'}],
  'text': 'Jane Doe'},
 {'findingDuration': None,
  'labels': [{'confidence': 0.8988,
              'detectedByRegex': False,
              'label': 8,
              'labelName': 'EmailAddress'}],
  'text': 'jane.doe@example.com'},
 {'findingDuration': None,
  'labels': [{'confidence': 0.9107,
              'detectedByRegex': False,
              'label': 35,
              'labelName': 'PhoneNumber'}],
  'text': '(415) 555-2671'},
 {'findingDuration': None,
  'labels': [{'confidence': 1.0,
              'detectedByRegex': True,
              'label': 39,
              'labelName': 'Ssn'}],
  'text': '123-45-6789'},
 {'findingDuration': None,
  'labels': [{'confidence': 1.0,
              'detectedByRegex': True,
              'label': 53,
              'labelName': 'CreditCard'}],
  'text': '4111 1111 1111 1111'},
 {'find

we can see above that the instance *Jane Doe*, does not have the categories 25:NameFamily and 26:NameGiven predicted (because they had scores lower than the thresholds we defined).

#### Passing One Global Threshold For All Categories

Another very useful way to experiment WITHOUT affecting future prediction is by passing *global_min_conf* - this threshold parameter will be applied to all categories and will not affect future results. 

See below -

In [19]:
text_result = client.detect_pii_in_text(text, human_readable_response=True, global_min_conf=0.0)
pprint(text_result.get('piiFindings', {}))

[{'findingDuration': None,
  'labels': [{'confidence': 0.9092,
              'detectedByRegex': False,
              'label': 24,
              'labelName': 'Name'},
             {'confidence': 0.4652,
              'detectedByRegex': False,
              'label': 25,
              'labelName': 'NameFamily'},
             {'confidence': 0.4496,
              'detectedByRegex': False,
              'label': 26,
              'labelName': 'NameGiven'}],
  'text': 'Jane Doe'},
 {'findingDuration': None,
  'labels': [{'confidence': 0.8988,
              'detectedByRegex': False,
              'label': 8,
              'labelName': 'EmailAddress'}],
  'text': 'jane.doe@example.com'},
 {'findingDuration': None,
  'labels': [{'confidence': 0.9107,
              'detectedByRegex': False,
              'label': 35,
              'labelName': 'PhoneNumber'}],
  'text': '(415) 555-2671'},
 {'findingDuration': None,
  'labels': [{'confidence': 1.0,
              'detectedByRegex': True,
          

Currently there is an open ticket to fix behavior of flag *per_label_min_conf* for the above enpoints. So, until notified please avoid using it.

#### Modify Category Thresholds back to Default

Finally, to avoid any unintended behavior in the future, lets set the category thresholds back to the default value of 0.0.

In [20]:
update_category_thresholds = {'25': 0.0, '26': 0.0}
client.post_thresholds(thresholds=update_category_thresholds)
print("categories updated")

current = client.get_thresholds()
print("Current thresholds (sample):")
[c for c in current['thresholds'] if current.get('tenantId') == client.tenant_id]

categories updated
Current thresholds (sample):


[{'label': 25, 'minimalConfidence': 0.0, 'labelName': 'NameFamily'},
 {'label': 26, 'minimalConfidence': 0.0, 'labelName': 'NameGiven'},
 {'label': 1, 'minimalConfidence': 0.75, 'labelName': 'AccountNumber'},
 {'label': 2, 'minimalConfidence': 0.75, 'labelName': 'Age'},
 {'label': 3, 'minimalConfidence': 0.75, 'labelName': 'Date'},
 {'label': 4, 'minimalConfidence': 0.75, 'labelName': 'DateInterval'},
 {'label': 5, 'minimalConfidence': 0.75, 'labelName': 'Dob'},
 {'label': 6, 'minimalConfidence': 0.75, 'labelName': 'DriverLicense'},
 {'label': 7, 'minimalConfidence': 0.75, 'labelName': 'Duration'},
 {'label': 8, 'minimalConfidence': 0.75, 'labelName': 'EmailAddress'},
 {'label': 9, 'minimalConfidence': 0.75, 'labelName': 'Event'},
 {'label': 10, 'minimalConfidence': 0.75, 'labelName': 'Filename'},
 {'label': 12, 'minimalConfidence': 0.75, 'labelName': 'HealthcareNumber'},
 {'label': 13, 'minimalConfidence': 0.75, 'labelName': 'IpAddress'},
 {'label': 14, 'minimalConfidence': 0.75, 'lab

## Summary

In this notebook, we demonstrated how to use the Votiro DDR Sensitive Data Detection & Masking API to identify and protect personally identifiable information (PII), protected health information (PHI), and payment card information (PCI).

You learned how to:

> - Detect sensitive data in both plain text and uploaded files.
> - Mask detected entities while preserving the structure and readability of the source content.
> - Retrieve and update tenant-specific thresholds to control model sensitivity and confidence levels.
> - Visualize original and masked documents side by side for validation.

These are playground capabilities by Votiro’s DDR (Data Detection and Redaction) pipeline, enabling secure data handling and compliance across diverse document formats.
For production use, ensure tokens are managed securely, tenant thresholds are configured to match your compliance needs, and only authorized users can access the endpoints.